# Building Kilosort4 in TorchBCI Using Modular Blocks

This notebook demonstrates how Kilosort4 is organized in TorchBCI using reusable modular blocks.

The goal is to show:
- how the Kilosort4 pipeline is decomposed into reusable stages
- how these stages map to shared framework abstractions
- how a complete algorithm can be built from standardized blocks

This notebook is intended as a tutorial example for building algorithms in TorchBCI at a higher level of abstraction.

## 1. Why modular blocks?

Instead of treating an algorithm as one monolithic implementation, TorchBCI organizes it into reusable blocks such as:

- preprocessing
- detection
- clustering
- saving/export

This makes the codebase easier to:
- read
- extend
- reuse across algorithms
- standardize under a shared framework design

In this notebook, Kilosort4 is used as an example of how an algorithm can be built from these blocks.

## 2. Kilosort4 block structure in TorchBCI

At a high level, the Kilosort4 pipeline in TorchBCI is composed of the following stages:

1. Initialize ops / configuration
2. Preprocessing
3. Drift correction
4. Spike detection
5. Clustering
6. Merge / cleanup
7. Save results

This decomposition makes it possible to reason about the algorithm at the framework level rather than only at the full-pipeline level.

In [ ]:
import inspect
from pprint import pprint

In [ ]:
from torchbci.block.base.base_block import BaseBlock
from torchbci.block.base.base_detection_block import BaseDetectionBlock
from torchbci.block.base.base_clustering_block import BaseClusteringBlock
from torchbci.block.base.base_save_block import BaseSaveBlock

## 3. Shared framework abstractions

The repository now includes shared abstract block categories such as:

- `BaseBlock`
- `BaseDetectionBlock`
- `BaseClusteringBlock`
- `BaseSaveBlock`

These classes do not change the algorithm logic. Their purpose is to introduce a higher-level structure so that algorithm-specific blocks can be grouped under common concepts.

In [ ]:
print('BaseBlock MRO:')
print(BaseBlock.__mro__)
print()

print('BaseDetectionBlock MRO:')
print(BaseDetectionBlock.__mro__)
print()

print('BaseClusteringBlock MRO:')
print(BaseClusteringBlock.__mro__)
print()

print('BaseSaveBlock MRO:')
print(BaseSaveBlock.__mro__)

## 4. Import Kilosort4 blocks

Next, we import the Kilosort-specific blocks used to assemble the Kilosort4 pipeline.

In [ ]:
from torchbci.block.kilosort_blocks.spikedetectionblock import SpikeDetectionBlock
from torchbci.block.kilosort_blocks.finalgraphclusteringblock import FinalGraphClusteringBlock
from torchbci.block.kilosort_blocks.saveresultsblock import SaveResultsBlock

In [ ]:
print('SpikeDetectionBlock -> BaseDetectionBlock:', issubclass(SpikeDetectionBlock, BaseDetectionBlock))
print('FinalGraphClusteringBlock -> BaseClusteringBlock:', issubclass(FinalGraphClusteringBlock, BaseClusteringBlock))
print('SaveResultsBlock -> BaseSaveBlock:', issubclass(SaveResultsBlock, BaseSaveBlock))

In [ ]:
print('SpikeDetectionBlock MRO:')
print(SpikeDetectionBlock.__mro__)
print()

print('FinalGraphClusteringBlock MRO:')
print(FinalGraphClusteringBlock.__mro__)
print()

print('SaveResultsBlock MRO:')
print(SaveResultsBlock.__mro__)

### Interpretation

The `issubclass(...)` checks confirm that the Kilosort-specific blocks are now organized under shared abstract categories.

The `__mro__` output shows the inheritance chain Python uses when resolving methods and attributes.

For example, if `SpikeDetectionBlock` has an MRO like:

`SpikeDetectionBlock -> BaseDetectionBlock -> BaseBlock -> nn.Module -> ...`

then this confirms that the block now belongs to the shared detection abstraction while still remaining a PyTorch module.

## 5. Mapping Kilosort4 stages to framework blocks

A useful way to think about Kilosort4 in TorchBCI is through a block mapping:

- **Detection stage** -> `SpikeDetectionBlock`
- **Clustering stage** -> `FinalGraphClusteringBlock`
- **Save/export stage** -> `SaveResultsBlock`

This pattern makes the pipeline easier to explain and reuse in a framework setting.

In [ ]:
print('SpikeDetectionBlock source header:')
print(inspect.getsource(SpikeDetectionBlock).splitlines()[0])
print()

print('FinalGraphClusteringBlock source header:')
print(inspect.getsource(FinalGraphClusteringBlock).splitlines()[0])
print()

print('SaveResultsBlock source header:')
print(inspect.getsource(SaveResultsBlock).splitlines()[0])

## 6. The full Kilosort4 pipeline

These blocks are then assembled inside the full `KS4Pipeline`.

The pipeline is responsible for:
- wiring the blocks together
- preserving the execution order
- exposing a complete algorithm interface

The important idea is that the algorithm can now be explained as a composition of reusable stages rather than only as one large implementation.

In [ ]:
from torchbci.algorithms.kilosort_ported import KS4Pipeline
print('KS4Pipeline imported successfully.')

In [ ]:
pipeline_source = inspect.getsource(KS4Pipeline)
print('\n'.join(pipeline_source.splitlines()[:80]))

### What to look for in `KS4Pipeline`

When reading the pipeline source, the main thing to notice is that the pipeline creates and wires together separate blocks for different stages.

This is the key framework idea:
- the pipeline defines the algorithm structure
- the blocks define the stage-specific behavior

## 7. Conceptual pipeline assembly

At a high level, the Kilosort4 pipeline can be thought of as:

```text
Initialize
  -> Preprocess
  -> Drift correction
  -> Detection
  -> Clustering
  -> Merge / cleanup
  -> Save
```

This decomposition is useful because the same framework pattern can later be reused for other algorithms as well.

In [ ]:
print('KS4Pipeline constructor signature:')
print(inspect.signature(KS4Pipeline.__init__))

## 8. Minimal example of pipeline construction

The following cell shows a minimal template for constructing the Kilosort4 pipeline.

This is provided for illustration only. It may require local data files, probe configuration, CUDA/CuPy, and other environment-specific dependencies to run fully.

In [ ]:
import torch

example_settings = {
    'n_chan_bin': 385,
    # 'fs': 30000,
    # 'filename': 'path/to/data.bin',
}

example_probe = {
    # Fill in with a real probe dictionary if available
    # 'xc': ...,
    # 'yc': ...,
    # 'kcoords': ...,
    # 'chanMap': ...,
}

# Example only:
# pipeline = KS4Pipeline(
#     settings=example_settings,
#     probe=example_probe,
#     results_dir='path/to/results',
#     device=torch.device('cuda' if torch.cuda.is_available() else 'cpu'),
# )
#
# print(pipeline)

## 9. Why this abstraction matters

The purpose of introducing shared block abstractions is not to change the algorithm itself.

Instead, the purpose is to make the framework:

- more standardized
- easier to extend
- easier to document
- easier to reuse across multiple algorithms

In other words, the Kilosort4 logic remains the same, but it is now expressed within a clearer framework structure.

## 10. Summary

In this notebook, we showed that:

- Kilosort4 in TorchBCI is organized as a modular pipeline
- specific Kilosort blocks now inherit from shared framework abstractions
- the new abstraction layer improves the structure of the repository without changing the underlying algorithm behavior

This makes Kilosort4 a concrete example of how algorithms can be built and standardized in TorchBCI using reusable blocks.

## Appendix: block mapping

| Stage | Kilosort block | Framework abstraction |
|---|---|---|
| Detection | `SpikeDetectionBlock` | `BaseDetectionBlock` |
| Clustering | `FinalGraphClusteringBlock` | `BaseClusteringBlock` |
| Save / export | `SaveResultsBlock` | `BaseSaveBlock` |

This table summarizes how algorithm-specific Kilosort components are mapped onto shared framework abstractions.